In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print("CUDA version:", torch.version.cuda)
print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
Device: cuda
CUDA version: 12.4
GPU name: NVIDIA GeForce RTX 2050


In [6]:

import torch

import cv2
from PIL import Image
import torch.nn as nn
import numpy as np
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import os

In [7]:
if torch.cuda.is_available():
    print("true")
 

true


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet=models.resnet50(pretrained=True)
resnet=nn.Sequential(*list(resnet.children())[:-1])
resnet=resnet.to(device)
resnet.eval()

c:\Users\karti\anaconda3\envs\pytorch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\karti\anaconda3\envs\pytorch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [9]:
x = torch.randn(1, 3, 224, 224).to("cuda")
resnet = resnet.to("cuda")
with torch.no_grad():
    out = resnet(x)
print(out.shape)


torch.Size([1, 2048, 1, 1])


In [10]:

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


In [11]:
class ImageFolderDataset(Dataset):
    def __init__(self, folder):
        self.paths = [os.path.join(folder, f) for f in os.listdir(folder)
                      if f.lower().endswith((".jpg", ".png", ".jpeg"))]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), path


In [13]:
image_folder =r"C:\Users\karti\OneDrive\Desktop\pytorch projects\pytorch project\Real-ESRGAN\myntradataset"
dataset = ImageFolderDataset(image_folder)
loader = DataLoader(dataset, batch_size=8, num_workers=0, pin_memory=True)
embeddings=[]
file_paths=[]

In [3]:

with torch.no_grad():
    for batch_imgs, batch_paths in loader:
        batch_imgs = batch_imgs.to(device)
        batch_embs = resnet(batch_imgs).squeeze()
        batch_embs = batch_embs.view(batch_embs.size(0), -1).cpu().numpy() 

        embeddings.extend(batch_embs)
        file_paths.extend(batch_paths)






NameError: name 'loader' is not defined

In [ ]:
import numpy as np

# Ensure all embeddings are 1D arrays of shape [2048]
fixed_embeddings = []
for emb in embeddings:
    emb = np.array(emb).flatten()  # flatten to [2048]
    if emb.shape[0] == 2048:
        fixed_embeddings.append(emb)
    else:
        print("Skipping malformed embedding with shape:", emb.shape)

fixed_embeddings = np.array(fixed_embeddings)
print("Final shape:", fixed_embeddings.shape)

np.save("embeddings.npy", fixed_embeddings)
np.save("file_paths.npy", np.array(file_paths[:len(fixed_embeddings)]))


Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with shape: (1,)
Skipping malformed embedding with 